# Draw In-Volume Signal Removed -- After the Cosmic Tagger Cut

The neutrinos the cosmic tagger cut **took away**: reco clusters that survived
the beam-window cut, matched an **in-volume** true neutrino, and were then
removed by the tagger cut.

This is the signal cost of that cut, interaction by interaction, with a BEE link
for every case so each one can be opened and judged.

## What is drawn

    survived the beam-window cut
    matched a true neutrino          match_reco_to_true_neutrino, purity >= 5%
    vertex_in_volume == True
    REMOVED by the cosmic tagger cut

**The TRUE cluster is on the top row, the RECO cluster below it**, the same order
every other pair population in this codebase uses. The rows share axes per
column, so what the cut took reads as the part of the lower row with no
counterpart above it.

## The tagger is run, not applied

`tag_reco_clusters` says which clusters the cut *would* delete, and the full
beam-window set is kept. Applying `apply_cosmic_tagger_cut` here would remove
exactly the population this notebook exists to draw.

The cut is PER-FLASH (selections.COSMIC_TAG_PROPAGATE_SCOPE = 'flash'), so a
cluster is here because a tagger flagged it, or because it shared that
cluster's FLASH. Activity at a different flash time in the event is kept. Each
which, and the distinction matters: the first is a reconstruction problem, the
second is the price of not being able to separate in-beam activity.

## Reading the numbers

The match bar is loose (5% purity), so a matched cluster can hold a fragment
rather than a whole interaction. `invol_signal_removed_summary.txt` therefore reports
**clusters and distinct interactions separately** -- two reco clusters can match
one neutrino -- and breaks the population down by how much of the neutrino each
cluster actually held. The interaction count is the one to quote.

## Output

    AnalysisDistributions/multi_file_plots_charge_light_matching/
        Draw_InVolumeSignal_Removed_After_CosmicTagger/
            combined_apa_<date>_<time>/
                InVolumeSignalRemoved/
                    invol_signal_removed_summary.txt   counts per channel
                    bee_links_numu_CC.txt         one link file per channel
                    bee_links_nue_CC.txt
                    bee_links_NC.txt
                    chunk0/
                        numu_CC/
                            invol_signal_removed_chunk0_event<n>_recoID<n>_trueID<n>.png
                        nue_CC/ ...
                        NC/ ...
                    chunk1/ ...

## BEE display

Each figure prints its event's BEE URL on one line below the bottom row of
panels. It is **text, not a link** -- PNG has no way to carry a hyperlink, so a
drawn-in link would be one that does not work. `bee_links.txt` in the same
directory holds the same URLs where they can be clicked or copied.

## Relationship to the other notebooks

The loading, cuts, cluster-id assignment, completeness/purity evaluation and
pair matching are byte-identical to `SignalBackground_Distributions.ipynb` --
this notebook is that one with the job-summary drawing phase removed and the
per-event view call redirected. Any change to the pipeline modules reaches both.
Nothing here writes into `Signal_Background_Distributions/`.


In [ ]:
# Run scope -- same knobs as Reco_Distributions.ipynb and
# Evaluation_ChargeLightMatching_AfterBeamWindowCut.ipynb. Charge-light matching
# is a combined-APA evaluation (img-global / sed-sce are already global across
# APAs) -- no per-APA/face looping.
files     = "all"   # "all", or 1/2/3/... to limit the number of file subdirectories processed
events    = "all"   # "all", or 1/2/3/... to limit the number of events processed per file

# ========================================================================
# SELECTIVE FILE/EVENT FILTERING (Optional)
# ========================================================================
# Set to None to process all files/events, or specify to run only specific ones
# Example: target_file = "file0", target_event = 3  (to process only file0, event 3)
target_file  = None   # Set to "file0", "file1", etc. to process specific file only
target_event = None   # Set to a SINGLE event number (0, 1, ..., 9); use target_event_range below for a span

# Range of event numbers to run, inclusive on both ends: (1, 5) runs events
# 1,2,3,4,5. None runs every event. Applied on top of target_event, so leave
# target_event = None when using a range.
target_event_range = None   # e.g. (1, 5) for events 1..5

# Range of file INDICES to run, inclusive on both ends: (6, 9) runs file6, file7,
# file8, file9. None runs every file. Matched on the number at the end of the
# directory name, NOT on position in the list -- the directories sort
# lexicographically (file0, file1, file10, file11, file2, ...), so a positional
# slice would pick the wrong files. The `files = N` knob above still takes the
# first N in lexicographic order.
# NOTE: target_file (above) is applied too, so set it to None when using a range,
# otherwise only the one file that satisfies both runs.
target_file_range = None   # e.g. (6, 9) for file6..file9

# Fail fast rather than silently processing nothing: `evt != target_event` can
# never be False for a tuple, so target_event = (1, 5) would skip every event.
if isinstance(target_event, (tuple, list)):
    raise ValueError(
        f"target_event={target_event} is a range, but target_event takes a single event number. "
        f"Use target_event_range={tuple(target_event)} and target_event = None instead.")


# ========================================================================
# NO LEVEL SWITCHES -- this notebook draws the JOB-LEVEL stack only.
# ========================================================================
# Unlike Reco_Distributions.ipynb there is no b_draw_event/file_level_plots
# knob. A stack answers "what is this sample made of", which needs the pooled
# statistics to mean anything: per event there are one or two signal clusters,
# so an event-level stack is a single bar of height one. The file/event loops
# below still run -- they are how the clusters are collected -- they just do not
# draw or write anything of their own.


In [ ]:
%load_ext autoreload
%autoreload 2

# python libraries
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import sys
import os
import time
from datetime import datetime

np.set_printoptions(linewidth=1000)

# This notebook lives in AnalysisDistributions/, one level below the
# repository root where the pipeline modules and the input trees are. Resolve
# both explicitly so the notebook runs whether Jupyter was started in this
# directory (the usual case) or at the repository root.
NB_DIR = Path.cwd()
if NB_DIR.name != "AnalysisDistributions":
    NB_DIR = NB_DIR / "AnalysisDistributions"
NB_DIR = NB_DIR.resolve()
REPO_ROOT = NB_DIR.parent

for path in (str(REPO_ROOT), str(NB_DIR)):
    if path not in sys.path:
        sys.path.insert(0, path)

# Record job start time (used to report total job runtime at the end)
job_start_time = time.time()
print(f"Job started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Notebook directory: {NB_DIR}")
print(f"Repository root:    {REPO_ROOT}")


In [ ]:
# Pipeline modules (repository root) -- imported and used UNCHANGED. Only the
# TRUE side of the chain is needed here (see the header): the reco clusters, the
# beam-window cut, completeness/purity and the 1-to-1 pairing are all absent
# because no quantity on these plots depends on them.
from readfiles import (ensure_data_extracted, stage_nuecc_chunks,
                       read_charge_light_files_for_event, flatten_mc_tree)
from selections import (
    GroupClustersByID, build_true_points_charge_light, apply_deadarea_cut_true_charge_light,
    reassign_cluster_ID_true_charge_light,
    apply_energy_cutoff, apply_true_pointwise_energy_cutoff,
    apply_energy_cutoff, apply_min_true_points_cutoff,
    apply_wire_readout_sensitive_yz_plane_cut_true,
    tag_reco_clusters,
)
from cluster_category import cluster_category
from completeness_purity_estimate import EvaluateCompleteness, EvaluatePurity
from clusterpairmatching import MatchTrueToReco1to1
from metadata import (
    build_true_cluster_type_records, build_neutrino_vertex_records,
    build_cluster_flash_metadata, build_img_cluster_flash_metadata,
    add_metadata_true_reco_pair_cluster,
)
from DrawRecoTrueClusterCount import DrawRecoClusterSelectionFlow
from DrawRecoTrueFlashes import (BEAM_WINDOW_MIN_US, BEAM_WINDOW_MAX_US,
                                 draw_clustering_flashes)

# Per-cluster record builder shared with Reco_Distributions.ipynb, so a cluster's
# total_energy means exactly the same thing in both notebooks.
from draw_variables import build_true_cluster_variable_records, build_reco_cluster_variable_records

# The new module for this notebook (AnalysisDistributions/draw_signal_background.py):
# the stack's component list, the channel join, the stacked drawer and the table.
from draw_signal_background import (
    SIGNAL_BACKGROUND_COMPONENTS, attach_interaction_channel, order_components_for_stack,
    draw_stacked_true_energy, write_signal_background_info, reco_cluster_energy_mev,
    shared_y_top, attach_pair_metrics, NUMU_QUALITY_THRESHOLD,
    write_signal_background_root, Y_SCALES, BIN_WIDTHS_MEV,
    SIGNAL_BACKGROUND_COMPONENTS_NUMU_QUALITY,
    RECO_WORK_FUNCTION_EV, RECO_RECOMBINATION_FACTOR, DEFAULT_RECO_CUTS_LABEL,
)

# Selection performance: reco-space categorisation and its three plots.
from draw_selection_performance import (
    match_reco_to_true_neutrino,
    categorize_reco_clusters, draw_reco_selection_stack, draw_completeness_vs_purity,
    draw_completeness_vs_purity_colz,
    build_selection_efficiency, draw_selection_efficiency,
    write_selection_performance_info, write_selection_performance_root,
    write_efficiency_summary, EFFICIENCY_CURVE_SETS,
    draw_neutrino_multiplicity, draw_efficiency_by_multiplicity, MULTIPLICITY_CLASSES,
    plot_directory, EFFICIENCY_THRESHOLDS, draw_efficiency_threshold_comparison,
    UNCERTAINTY_STYLES, UNCERTAINTY_DIRS, count_signal_interactions_per_event,
    draw_energy_reconstruction, ENERGY_RECO_QUALITY,
    MULTIPLICITY_FIGURES,
    EFFICIENCY_BINNINGS, rebin_efficiency_tail,
    CHANNELS, MIN_MATCH_PURITY, HIGH_SIGNAL_THRESHOLD,
)

# load_bee_links comes from the contamination module: the BEE links file and
# its per-event URL rule are the same for both populations, and two readers of
# one file would drift.
from draw_contamination_clusters import load_bee_links
from draw_tagger_impact import (
    IN_VOLUME_SIGNAL_REMOVED_DIR_NAME, OUT_OF_VOLUME_SURVIVED_DIR_NAME, CHANNELS,
    save_event_tagger_impact, write_bee_links_by_category, write_impact_summary,
)


In [ ]:
# Configuration: Parent directory containing multiple file subdirectories (file0/, file1/, ...)
#
# Expected structure (per file subdirectory). The preprocessed tree has no zip --
# its data/ is already present, so ensure_data_extracted() below simply no-ops.
# PARENT_DIR/
#   file0/data/0/0-sed-smear_readout.json                (true clusters)
#   file0/data/0/0-mc.json                               (particle truth ancestry tree)
#   file0/data/1/, 2/, ... (one subdirectory per event)
#
# The reco/optical files in the same tree (0-img-global.json,
# 0-clustering-global.json, 0-op.json) are read by read_charge_light_files_for_event
# but not used here -- this notebook is truth only.

# ========================================================================
# INPUT SAMPLE  (kept identical to SignalBackground_Distributions.ipynb)
# ========================================================================
#   "tagger_100files" -> the MCP2025C Fall, Tagger-included 100-file production
#       (chunk0../chunk9, data/ already present; RAW -- dead-area cut not applied).
#   "nuecc" -> the img-clus-match-tag-pr-nuecc sample. Its 8866 zips are split
#       into bee/chunk_00 .. chunk_88, each into subchunk_00.. of 10. A job runs
#       every chunk in NUECC_SOURCE_CHUNKS; stage_nuecc_chunks() rewrites each
#       chunk's subchunks into staging/<chunk>__<subchunk>/data/<k>/ -- the same
#       <file>/data/<event>/ layout this loop expects.
SAMPLE = "nuecc"

NUECC_BEE_ROOT      = Path("/Volumes/My Passport/Research_Life/Experiment/SBND/"
                           "Wirecell_Reconstruction/Samples/"
                           "img-clus-match-tag-pr-nuecc-1000file-2026-08-29/bee")
NUECC_SOURCE_CHUNKS = [f"chunk_{i:02d}" for i in range(10)]   # bee/chunk_NN dir(s) this job runs
NUECC_STAGING_ROOT  = NUECC_BEE_ROOT.parent / "staging"
NUECC_N_FILES       = 100   # per source chunk (100 = the whole chunk)
NUECC_CHUNK_SIZE    = 10    # events per staged subchunk

if SAMPLE == "nuecc":
    PARENT_DIR     = NUECC_STAGING_ROOT
    BEE_LINKS_FILE = REPO_ROOT / "bee_links_nuecc_chunk.txt"
else:
    PARENT_DIR     = REPO_ROOT / "Haiwang_files_charge_light_matching_Tagger_Included_MCP2025C_FallProd_100files"
    BEE_LINKS_FILE = REPO_ROOT / "bee_links_100_files_chunk.txt"

# Number of files/events to process (convert the 'files'/'events' knobs above)
num_files_to_process  = None if files  == "all" else files
num_events_to_process = None if events == "all" else events

# Output directory: inside AnalysisDistributions, alongside
# RecoTrue_Distributions_AfterTimeWindowCut, so each notebook in this directory
# owns one subdirectory of the same plot tree.
PLOTBASEDIR = NB_DIR / "multi_file_plots_charge_light_matching" / "Draw_InVolumeSignal_Removed_After_CosmicTagger"
PLOTBASEDIR.mkdir(parents=True, exist_ok=True)

print("Configuration:")
print(f"Parent directory: {PARENT_DIR}")
print(f"Plot base directory: {PLOTBASEDIR}")
print(f"Files to process: {files}")
print(f"Events to process: {events}")

if target_file is not None or target_event is not None or target_file_range is not None or target_event_range is not None:
    print(f"\nSELECTIVE FILTERING ENABLED:")
    print(f"  Target file: {target_file if target_file else 'all'}")
    print(f"  Target event: {target_event if target_event is not None else 'all'}")
    if target_event_range is not None:
        print(f"  Target event range: event{target_event_range[0]}..event{target_event_range[1]} (inclusive)")
    if target_file_range is not None:
        print(f"  Target file range: file{target_file_range[0]}..file{target_file_range[1]} (inclusive)")

# ========================================================================
# SELECTION PARAMETERS -- the TRUE-side subset of
# Evaluation_ChargeLightMatching_AfterBeamWindowCut.ipynb's, deliberately
# identical: these distributions describe the true population that notebook
# evaluates, so any change here breaks that correspondence.
#
# The reco-side parameters (matching radii, min_reco_points_cutoff, the
# beam-window cut) are not here because no reco cluster is read.
# ========================================================================

# min_true_points_cutoff is DISABLED: this format's point clouds are much
# sparser than the old imaging-based reconstruction -- real neutrino clusters
# have been seen with as few as 13 points -- so the old threshold (200) would
# delete real signal clusters outright.
#
# min_cluster_energy IS applied: sed-smear's per-point 'e' field (MeV) is a
# genuine energy deposit, so the old threshold (100 MeV) carries over directly.
# NOTE it also sets where this histogram's populated range begins -- no surviving
# cluster can fall in the 0-100 MeV bin.
# Matching radii for the 1-to-1 pairing, identical to
# Evaluation_ChargeLightMatching_AfterBeamWindowCut.ipynb. Needed again now that
# the NumuCCQuality plot splits the signal by the completeness and purity of
# each cluster's match -- change one of these and that split moves.
radius_completeness       = 2
radius_purity_xz          = 3
radius_purity_yz          = 5
radius_purity_xy          = 5

# HOW MANY of the three projected distances a reco point must satisfy to count
# as matched. 3 = the historic all-three rule; 2 = any two, so no single cut can
# veto a match on its own.
#
# With 2 and radius_purity_xz raised 2 -> 3, measured over the full sample
# (553 pairs, 1.29M points): ~43,000 more points accepted, 16 pairs promoted
# into the high-signal region, and only 2 of 553 pairings repoint to a
# different true cluster -- so completeness moves very little. See
# Purity_Cut_Geometry/ for the study.
purity_min_projections    = 2
min_recopoints_threshold  = 5

min_cluster_energy        = 100     # APPLIED (Apply_energy_cutoff = True below)
min_true_point_energy     = 0.02    # MeV per POINT (Apply_trueenergy_pointwise_cutoff below)
min_true_points_cutoff    = 200     # NOT APPLIED (Apply_min_true_points_cutoff = False below)

Apply_energy_cutoff                         = True
Apply_trueenergy_pointwise_cutoff           = True    # drop true POINTS below min_true_point_energy
Apply_min_true_points_cutoff                = False
Apply_wire_readout_sensitive_xz_plane_cut   = True
Apply_time_window_cut                       = False   # must stay disabled -- no per-point true time in this format
# The dead-area cut is APPLIED, just not here: PARENT_DIR above is the tree
# preprocess_deadarea_cut.py already cut. Set this True only if you point
# PARENT_DIR back at a raw tree.
Apply_deadarea_cut                          = False

# The tagger is RUN but nothing is removed: this notebook needs to know which
# clusters the cut would delete, so it keeps the full beam-window set and asks
# tag_reco_clusters. Applying the cut would empty the population.
Apply_cosmic_tagger_cut                     = True

# FIDUCIAL volume (unit: cm) -- the SIGNAL definition. These are the bounds
# for the vertex_in_volume flag, and that is ALL they are: an interaction
# counts as signal when its VERTEX sits inside this volume.
#
# They are deliberately NOT applied to the true points. The points keep
# every deposit the wire-readout cut left, because a neutrino that starts
# inside the fiducial volume and throws tracks past its edge is still that
# neutrino, and the energy it put outside still belongs to it. Cutting the
# points here would quietly shrink the true energy of exactly the
# interactions nearest the boundary.
#
# Taken from selections.py rather than written out here so the signal
# definition lives in one place.
from selections import (Fiducial_X_MIN, Fiducial_X_MAX,
                        Fiducial_Y_MIN, Fiducial_Y_MAX,
                        Fiducial_Z_MIN, Fiducial_Z_MAX)
x_min, x_max = Fiducial_X_MIN, Fiducial_X_MAX
y_min, y_max = Fiducial_Y_MIN, Fiducial_Y_MAX
z_min, z_max = Fiducial_Z_MIN, Fiducial_Z_MAX

# Metadata label only -- there is no 2-view/3-view distinction in the
# charge-light format, so this is just a constant.
view = "combined"

# ========================================================================
# RECO OVERLAY
# ========================================================================
# One plot is drawn PER RECO SELECTION. The stack is identical in all of them --
# the true side is not touched -- so what each plot compares is the same true
# population against a differently-cut reco population.
#
# The label goes in the plot title, the legend and BOTH output filenames, so the
# versions sit side by side in one directory instead of overwriting each other.
#
# KEEP THE LABELS HONEST: nothing checks that a label matches the cuts the main
# loop actually applies, and a plot labelled with the wrong selection is worse
# than no plot. Adding a selection means adding it here AND building its cluster
# dict in the loop below.
RECO_SELECTION_NOCUTS = DEFAULT_RECO_CUTS_LABEL       # 'NoCuts': every reco cluster
RECO_SELECTION_BEAM   = 'AfterBeamWindowCut'          # flash time inside the beam window
RECO_SELECTION_LABELS = [RECO_SELECTION_NOCUTS, RECO_SELECTION_BEAM]

# WHICH TRUTH FILE the true clusters come from.
#
#   True  -> sed-sce_smear_readout : true positions WITH the space-charge
#            displacement, which is what clustering-global's reco positions
#            carry. Matching 225,633 reco points to their nearest true point
#            over 8 events gives a median residual of 0.394 cm against this
#            file and 0.555 cm against sed-smear, so this is the variant the
#            reco actually sits on.
#   False -> sed-smear_readout : true positions, no space charge. What every
#            run before 2026-08-14 used.
#
# Only x/y/z differ between the two -- every energy, cluster id and nu_idx is
# identical -- so this moves completeness and purity and nothing else.
TRUE_SOURCE_SCE     = True

# WHICH ID FIELD defines a reco cluster in clustering-global.
#
#   'cluster_id'      -> the COARSE grouping: everything the charge-light
#            matching tied to one flash counts as ONE reco cluster. Measured
#            over 1363 events, 50 of 51 groups of beam-window clusters that
#            share a flash time sit inside a single cluster_id, so this is
#            very nearly 'group all in-beam activity together'.
#   'real_cluster_id' -> the FINER grouping, used by every run before
#            2026-08-15. It keeps apart pieces that cluster_id merges,
#            including, in 13 of those 51 groups, two genuinely different
#            neutrinos.
#
# The two fields are identical in img-global and in the truth files; the split
# exists only in clustering-global.
RECO_ID_FIELD       = 'cluster_id'

# The figures to draw: (component list, variant label).
#
# TWO, not three. There used to be one figure per reco selection, because each
# carried a reco overlay. With the overlay gone the stack is pure truth and does
# not depend on the reco selection at all, so a 'NoCuts' and an
# 'AfterBeamWindowCut' version of the plain stack would be the same picture drawn
# twice under two names -- and two names implying a difference that is not there
# is worse than one name.
#
# The second splits the signal band by how well each cluster was reconstructed:
# completeness AND purity above NUMU_QUALITY_THRESHOLD, or not. That one DOES
# depend on the reco side -- the 1-to-1 pairing behind those numbers is computed
# against the beam-window reco clusters -- but the dependence is in how the true
# clusters are split, not in anything drawn on top of them.
PLOT_VARIANTS = [
    (None, None),
    # (SIGNAL_BACKGROUND_COMPONENTS_NUMU_QUALITY, 'NumuCCQuality'),
]

# The quality split is the ONLY thing here that needs completeness and purity,
# and computing them (EvaluateCompleteness + EvaluatePurity + MatchTrueToReco1to1
# per event) is about half this notebook's runtime. So the pairing is run only
# when a variant asks for it -- uncommenting the line above turns it back on by
# itself, with no second switch to remember.
#
# The SELECTION PERFORMANCE plots (below) also need it: every one of them is
# built on the reco-true pairing. So the pairing runs whenever either family of
# plots asks for it.
b_draw_selection_performance = True

NEEDS_PAIRING = b_draw_selection_performance or any(
    components is SIGNAL_BACKGROUND_COMPONENTS_NUMU_QUALITY
    for components, _ in PLOT_VARIANTS)

print("\nCuts applied (true side):")
if Apply_energy_cutoff:
    print(f"- Energy cutoff applied AFTER the volume cuts (threshold {min_cluster_energy} MeV "
          f"of SURVIVING energy, using sed-smear's per-point 'e' field)")
if Apply_trueenergy_pointwise_cutoff:
    print(f"- True POINT-wise energy cutoff applied BEFORE it (threshold {min_true_point_energy} MeV per point)")
if Apply_wire_readout_sensitive_xz_plane_cut:
    print(f"- Wire readout sensitive xz plane cut applied")
if Apply_deadarea_cut:
    print(f"- Dead area cut applied HERE")
else:
    print(f"- Dead area cut applied UPSTREAM by preprocess_deadarea_cut.py (baked into PARENT_DIR)")
print(f"- FIDUCIAL bounds (vertex_in_volume only -- NOT a cut on the true points): x [{x_min}, {x_max}], y [{y_min}, {y_max}], z [{z_min}, {z_max}] cm")

print(f"\nReco selections counted (not plotted): " + ", ".join(RECO_SELECTION_LABELS))
print(f"  beam window: {BEAM_WINDOW_MIN_US} - {BEAM_WINDOW_MAX_US} us (flash time)")
print(f"  reco energy = {RECO_WORK_FUNCTION_EV} eV * charge / {RECO_RECOMBINATION_FACTOR}")

print("\nStack components (bottom first):")
for component in SIGNAL_BACKGROUND_COMPONENTS:
    print(f"- {component['key']}")

# ========================================================================
# ONE-TIME EXTRACTION  (idempotent -- a populated data/<k>/ or data/ is left alone)
# ========================================================================
if SAMPLE == "nuecc":
    staged_chunks = []
    for source_chunk in NUECC_SOURCE_CHUNKS:
        print(f"Staging {NUECC_N_FILES} zips from {NUECC_BEE_ROOT / source_chunk} "
              f"in subchunks of {NUECC_CHUNK_SIZE}")
        staged_chunks += stage_nuecc_chunks(
            NUECC_BEE_ROOT / source_chunk, NUECC_STAGING_ROOT,
            n_files=NUECC_N_FILES, chunk_size=NUECC_CHUNK_SIZE)
    print(f"Staged {len(staged_chunks)} subchunk dir(s) from "
          f"{len(NUECC_SOURCE_CHUNKS)} source chunk(s)")
elif PARENT_DIR.exists():
    for subdir in sorted(PARENT_DIR.iterdir()):
        if subdir.is_dir():
            ensure_data_extracted(subdir)
else:
    print(f"Error: Parent directory {PARENT_DIR} does not exist")


In [ ]:
def find_all_input_directories(parent_dir):
    """
    Scan parent directory for all subdirectories containing 'data' folder.
    Returns a list of file directories (file0/, file1/, etc.).
    """
    parent_dir = Path(parent_dir)
    if not parent_dir.exists():
        print(f"Error: Parent directory {parent_dir} does not exist")
        return []

    data_dirs = []
    for subdir in sorted(parent_dir.iterdir()):
        if subdir.is_dir():
            data_path = subdir / "data"
            if data_path.exists() and data_path.is_dir():
                data_dirs.append(subdir)
                print(f"Found: {subdir}")

    return data_dirs


def file_index_from_name(name):
    """
    Trailing integer of a file directory name ("file10" -> 10), or None if it
    has no trailing digits. Used by target_file_range so files are selected by
    their real index rather than by position in the lexicographically sorted
    list (file0, file1, file10, file11, file2, ...).
    """
    digits = ""
    for ch in reversed(name):
        if not ch.isdigit():
            break
        digits = ch + digits
    return int(digits) if digits else None


def detect_events_in_directory(input_dir):
    """
    Auto-detect the number of events in a directory.
    Events are identified as numeric subdirectories in data/.
    Returns a sorted list of event numbers.
    """
    input_dir = Path(input_dir)
    data_dir = input_dir / "data"

    if not data_dir.exists():
        print(f"Warning: Data directory {data_dir} does not exist")
        return []

    events = []
    for item in data_dir.iterdir():
        if item.is_dir():
            try:
                events.append(int(item.name))
            except ValueError:
                pass

    return sorted(events)


# Auto-detect all input directories from parent directory
print(f"Scanning parent directory: {PARENT_DIR}")
print("-" * 60)
input_directories = find_all_input_directories(PARENT_DIR)
if SAMPLE == "nuecc":
    # keep only the staged dirs this run wrote (staging/ may hold others)
    staged_names = {d.name for d in staged_chunks}
    input_directories = [d for d in input_directories if d.name in staged_names]
if num_files_to_process is not None:
    input_directories = input_directories[:num_files_to_process]
else:
    num_files_to_process = len(input_directories)
print("-" * 60)

print(f"\nFound {len(input_directories)} input directories with data/\n")
if input_directories:
    for input_dir in input_directories:
        detected_events = detect_events_in_directory(input_dir)
        if detected_events:
            print(f"  {input_dir.name}/data/: {len(detected_events)} events ({min(detected_events)}-{max(detected_events)})")
        else:
            print(f"  {input_dir.name}/data/: No events found")
else:
    print(f"Error: No subdirectories with 'data/' found in {PARENT_DIR}")


In [ ]:
# ============================================================================
# MAIN PROCESSING LOOP -- combined-APA signal/background distributions
# ============================================================================
# The true-side selection chain below is Reco_Distributions.ipynb's, unchanged:
# sed-smear_readout grouped by REAL_CLUSTER_ID and reassigned to 99990+nu_idx
# (neutrino, one cluster per interaction) / avg-X (cosmic), then the energy and
# fiducial cuts. The reco half of that loop is deliberately absent -- see the
# header.
#
# Only the JOB-LEVEL stack is drawn (see the header): the loops below collect
# clusters, they do not write per-file or per-event output.
#
# Each event contributes two things to the stack: the true cluster variable
# records (which carry total_energy and vertex_in_volume) and the mc.json vertex
# records (which carry interaction_channel). attach_interaction_channel joins the
# second onto the first by (event, cluster id), and the component selectors in
# draw_signal_background.py read both fields off the finished records.

timestamp  = datetime.now().strftime("%Y%m%d_%H%M%S")
# One directory per run, as the other notebooks do: the selection window and
# the pipeline underneath it change, and a figure is only interpretable
# against the run that produced it. Re-running leaves the previous set
# untouched beside this one.
# One directory per run under PLOTBASEDIR; the module puts everything it writes
# inside IN_VOLUME_SIGNAL_REMOVED_DIR_NAME within it: the bee-link files and the summary sit there,
# one directory above the per-chunk figure directories.
output_dir = PLOTBASEDIR / f"combined_apa_{timestamp}"
output_dir.mkdir(parents=True, exist_ok=True)
print(f"\n{'='*70}")
print(f"Output directory: {output_dir}")
print(f"{'='*70}\n")

job_true_var_records     = []   # one record per selected true cluster, channel attached
job_img_cluster_flash_records = []   # flashes of ALL reco clusters, before any cut

# Reco cluster selection flow: how many reco clusters survive each stage. Same
# two stages SelectionAnalysis.ipynb uses, but counted in whichever namespace
# RECO_ID_FIELD selects -- with 'cluster_id' the totals are the coarse clusters
# this notebook actually analyses, not the finer real_cluster_id ones.
RECO_FLOW_STAGES = [(RECO_SELECTION_NOCUTS, 'No cuts'),
                    (RECO_SELECTION_BEAM,   '+ Beam window')]
job_reco_flow_counts = {label: 0 for label, _ in RECO_FLOW_STAGES}
job_reco_var_records     = {label: [] for label in RECO_SELECTION_LABELS}  # counted, and the beam-window set feeds the pairing
# Fresh per job so a re-run fills the same view slots again rather than silently
# drawing nothing the second time.
# One entry per figure. The two denominators make the counts readable: how many
# clusters survived the beam-window cut at all, and how many of those matched a
# true neutrino of any kind.
invol_removed_entries = []
n_beam_window_clusters = 0
n_matched_neutrino     = 0
bee_links = load_bee_links(BEE_LINKS_FILE)
print(f"BEE links loaded for {len(bee_links)} chunk(s)")
job_selection_records    = []   # one per selected reco cluster, with its category
job_cluster_type_records = []   # per-true-cluster is_neutrino (build_true_cluster_type_records)
job_vertex_records       = []   # per true neutrino interaction (build_neutrino_vertex_records)
total_events_processed   = 0
total_files_processed    = 0

for file_idx, input_dir in enumerate(input_directories):
    input_file_name = input_dir.name

    # SELECTIVE FILTERING: Skip files that don't match target_file
    if target_file is not None and input_file_name != target_file:
        print(f"Skipping {input_file_name} (target: {target_file})")
        continue

    # SELECTIVE FILTERING: Skip files outside target_file_range (inclusive both
    # ends, matched on the directory name's trailing index -- see
    # file_index_from_name). Applied on top of target_file, not instead of it.
    if target_file_range is not None:
        file_idx = file_index_from_name(input_file_name)
        range_low, range_high = target_file_range
        if file_idx is None or not (range_low <= file_idx <= range_high):
            print(f"Skipping {input_file_name} (target range: file{range_low}..file{range_high})")
            continue

    print(f"\n{'='*70}")
    print(f"FILE {file_idx+1}/{len(input_directories)}: {input_dir}")
    print(f"{'='*70}")

    events_list = detect_events_in_directory(input_dir)
    if not events_list:
        print(f"No events found in {input_dir}, skipping...")
        continue

    event_low = min(events_list)
    event_high = max(events_list) + 1 if num_events_to_process is None else event_low + num_events_to_process

    print(f"Processing events {event_low} to {event_high-1}\n")
    total_files_processed += 1

    # Start of event loop
    for evt in range(event_low, event_high):
        # SELECTIVE FILTERING: Skip events that don't match target_event
        if target_event is not None and evt != target_event:
            continue

        # SELECTIVE FILTERING: Skip events outside target_event_range (inclusive
        # both ends). Applied on top of target_event, not instead of it.
        if target_event_range is not None:
            event_range_low, event_range_high = target_event_range
            if not (event_range_low <= evt <= event_range_high):
                continue

        result = read_charge_light_files_for_event(input_dir, evt)
        if result is None:
            print(f"  Event {evt}: could not read data, skipping")
            continue

        event_key = f"{input_file_name}_{evt}"

        true_key = 'true_clustering_sce' if TRUE_SOURCE_SCE else 'true_clustering'
        if result.get(true_key) is None:
            raise RuntimeError(f"event {event_key}: {true_key} missing -- "
                               f"sed-sce_smear_readout.json is absent for this event")
        x_true, y_true, z_true, id_true, q_true, real_id_true, e_true, nu_idx_true = result[true_key]
        x_clu,  y_clu,  z_clu,  id_clu,  q_clu,  real_id_clu                       = result['clustering']
        mc_tree = result['mc']
        op_data = result['op']

        # ------------------------------------------------------------------
        # TRUE POINTS: the chosen truth variant in the standard 7-column shape
        # (energy = per-point 'e' in MeV; q_true = 'nu_idx', 0=cosmic,
        # 1/2/...=which neutrino interaction), reassigned to 99990+nu_idx
        # (neutrino) / avg-X (cosmic), then cut.
        # real_id_true (real_cluster_id), NOT id_true: cluster_id is a coarser
        # grouping that can merge physically distinct tracks.
        # ------------------------------------------------------------------
        true_points = build_true_points_charge_light(
            x_true, y_true, z_true, real_id_true, q_true, energy=e_true, nu_idx=nu_idx_true)
        true_points = reassign_cluster_ID_true_charge_light(true_points)

        # Snapshot BEFORE the cuts: build_neutrino_vertex_records uses it to say
        # what a removed neutrino actually deposited. Grouping only, no filtering.
        clusters_true_precut = GroupClustersByID(true_points)

        # CUT ORDER: every POINT-level cut runs before every CLUSTER-level one.
        #
        # The fiducial and dead-area cuts delete individual points; the energy
        # and min-point cuts test a whole cluster and keep or drop it. Running a
        # cluster-level test first means testing a quantity the later point-level
        # cuts then change -- and that is not hypothetical: with the energy cut
        # first, a cluster admitted at 116 MeV came out of the fiducial cut with
        # 64 MeV of surviving points and was plotted BELOW the 100 MeV threshold
        # it had supposedly passed. Out-of-volume neutrinos showed it worst, most
        # of their deposit being outside the volume by construction.
        #
        # In this order min_cluster_energy means 100 MeV of energy that SURVIVED
        # the cuts -- the same quantity the histograms are filled with -- so
        # nothing can appear below the threshold.
        if Apply_wire_readout_sensitive_xz_plane_cut:
            true_points = apply_wire_readout_sensitive_yz_plane_cut_true(true_points)
        if Apply_deadarea_cut:
            # The only thing that writes per-event output, and only when the cut
            # is run HERE rather than upstream -- so its directory is created
            # here rather than for every event of every run.
            deadarea_dir = output_dir / input_file_name / f"event_{evt:03d}"
            deadarea_dir.mkdir(parents=True, exist_ok=True)
            true_points = apply_deadarea_cut_true_charge_light(true_points, output_dir=deadarea_dir, event=evt, file_name=input_file_name)
        if len(true_points) == 0:
            print(f"  Event {evt}: no true points survive the volume cuts, skipping")
            continue
        # POINT-wise first, so the cluster total the cluster cut tests is the
        # total of the points that survive.
        if Apply_trueenergy_pointwise_cutoff:
            true_points = apply_true_pointwise_energy_cutoff(true_points, min_true_point_energy)
        if Apply_energy_cutoff:
            true_points = apply_energy_cutoff(true_points, min_cluster_energy)
        if Apply_min_true_points_cutoff:
            true_points = apply_min_true_points_cutoff(true_points, min_true_points_cutoff)

        if len(true_points) == 0:
            print(f"  Event {evt}: no true points remain after cuts, skipping")
            continue

        clusters_true = GroupClustersByID(true_points)

        # ------------------------------------------------------------------
        # TRUTH LABELS: neutrino/cosmic per cluster, and the mc.json interaction
        # vertices joined to their true cluster by nu_idx (cluster_id =
        # 99990+nu_idx, an exact key). vertex_in_volume uses the same bounds as
        # the fiducial cut; interaction_channel is numu_CC / nue_CC / NC.
        # ------------------------------------------------------------------
        event_cluster_type_records = build_true_cluster_type_records(
            clusters_true, input_file_name, evt, event_key)
        event_vertex_records = build_neutrino_vertex_records(
            flatten_mc_tree(mc_tree), clusters_true, input_file_name, evt, event_key,
            x_min=x_min, x_max=x_max, y_min=y_min, y_max=y_max, z_min=z_min, z_max=z_max,
            clusters_true_precut=clusters_true_precut, min_cluster_energy=min_cluster_energy)

        # ------------------------------------------------------------------
        # CLUSTER RECORDS + DRAWING (draw_signal_background.py)
        # ------------------------------------------------------------------
        event_true_var_records = build_true_cluster_variable_records(
            clusters_true, input_file_name, evt, event_key, "Combined",
            vertex_records=event_vertex_records)
        attach_interaction_channel(event_true_var_records, event_vertex_records)

        # ------------------------------------------------------------------
        # RECO CLUSTERS. Not plotted -- counted for summary.txt, and the
        # beam-window set is what the 1-to-1 pairing runs against.
        # clustering-global (post charge-light matching) grouped by
        # REAL_CLUSTER_ID, and nothing else: no beam-window cut, no fiducial
        # cut, no minimum point count. That is what RECO_CUTS_LABEL = 'NoCuts'
        # names, and adding a cut here means changing that label too.
        #
        # real_cluster_id, not cluster_id: cluster_id is a coarser grouping that
        # can merge physically distinct tracks (same reasoning as the true side).
        # reassign_cluster_ID_reco is deliberately NOT called -- it relabels
        # clusters by rounded average X, which MERGES clusters that happen to
        # share one, and this plot counts reco clusters.
        # ------------------------------------------------------------------
        reco_ids_clu = id_clu if RECO_ID_FIELD == 'cluster_id' else real_id_clu
        predicted_points = np.column_stack((x_clu, y_clu, z_clu, reco_ids_clu, q_clu))

        # BEAM-WINDOW CUT, the second selection. op.json flashes are attached to
        # img-global clusters and then bridged onto clustering-global clusters by
        # point charge; a reco cluster passes if its bridged flash time falls in
        # [BEAM_WINDOW_MIN_US, BEAM_WINDOW_MAX_US]. Identical to the cut
        # Evaluation_ChargeLightMatching_AfterBeamWindowCut.ipynb applies, and it
        # reads the same two constants, so widening the window there widens it here.
        #
        # build_img_cluster_flash_metadata keys its records by REAL_CLUSTER_ID
        # (deliberately -- see metadata.py). When RECO_ID_FIELD is 'cluster_id'
        # the points are grouped in a DIFFERENT namespace, so the beam-window ids
        # must be translated or np.isin below matches nothing and every event
        # comes out empty. A cluster_id passes if any of its real sub-clusters
        # has a beam-window flash, which is what treating them as one activity
        # means.
        #
        # A cluster with NO bridged flash is CUT: it has no time, so it cannot be
        # shown to be in the beam window. That is the evaluation's behaviour too.
        event_flash_metadata_list = build_cluster_flash_metadata(
            op_data, input_file_name, evt, "Combined", event_key)
        event_img_cluster_flash_records = build_img_cluster_flash_metadata(
            result['reco'], result['clustering'], event_flash_metadata_list,
            input_file_name, evt, "Combined", event_key)
        clu_beam_window_ids = {float(r['clustering_cluster_id']) for r in event_img_cluster_flash_records
                               if BEAM_WINDOW_MIN_US <= r['flash_time'] <= BEAM_WINDOW_MAX_US}
        real_to_coarse = {float(r): float(c) for r, c in zip(real_id_clu, id_clu)}
        if RECO_ID_FIELD == 'cluster_id':
            clu_beam_window_ids = {real_to_coarse[r] for r in clu_beam_window_ids
                                   if r in real_to_coarse}

        # A RELABELLED copy for the flash plot. build_img_cluster_flash_metadata
        # keys by real_cluster_id on purpose (see metadata.py), but the flash plot
        # should count the same objects the stacks do -- otherwise the two
        # disagree by however many flash-mates the coarse grouping merged (137 vs
        # 104 on chunk0). draw_clustering_flashes deduplicates on
        # (event, cluster id, flash_index), so relabelling collapses merged
        # flash-mates automatically. The SELECTION above still uses the original
        # records; only this copy is relabelled.
        if RECO_ID_FIELD == 'cluster_id':
            event_flash_records_for_plot = [
                {**r, 'clustering_cluster_id': real_to_coarse.get(
                    float(r['clustering_cluster_id']), r['clustering_cluster_id'])}
                for r in event_img_cluster_flash_records]
        else:
            event_flash_records_for_plot = event_img_cluster_flash_records

        if len(predicted_points) and clu_beam_window_ids:
            beam_ids_array = np.fromiter(clu_beam_window_ids, dtype=float, count=len(clu_beam_window_ids))
            predicted_points_beam = predicted_points[np.isin(predicted_points[:, 3], beam_ids_array)]
        else:
            predicted_points_beam = predicted_points[:0]

        event_reco_points_by_selection = {
            RECO_SELECTION_NOCUTS: predicted_points,
            RECO_SELECTION_BEAM:   predicted_points_beam,
        }
        event_reco_var_records = {}
        event_reco_clusters    = {}
        for selection_label, selection_points in event_reco_points_by_selection.items():
            selection_clusters = GroupClustersByID(selection_points) if len(selection_points) else {}
            event_reco_clusters[selection_label] = selection_clusters
            event_reco_var_records[selection_label] = build_reco_cluster_variable_records(
                selection_clusters, input_file_name, evt, event_key, "Combined")

        # ------------------------------------------------------------------
        # 1-TO-1 TRUE-RECO PAIRING, for the NumuCCQuality split only.
        # Completeness and purity are computed here because that split needs a
        # per-cluster quality, and MatchTrueToReco1to1 needs both to choose a
        # pair. Nothing else on any of these plots depends on the pairing.
        #
        # Paired against the BEAM-WINDOW reco clusters, not the uncut ones: that
        # is the population the plot showing this split is drawn on, and it is
        # the same pairing Evaluation_ChargeLightMatching_AfterBeamWindowCut.ipynb
        # performs, with the same radii. Pairing against the uncut set would
        # score each true cluster against reco clusters the plot does not show.
        # ------------------------------------------------------------------
        event_pair_metadata_list = []
        completeness_results = purity_results = []
        if NEEDS_PAIRING:
         clusters_reco_for_pairing = event_reco_clusters[RECO_SELECTION_BEAM]
         cluster_category_results = cluster_category(
             clusters_true, output_dir=None, event=evt, apa="Combined", file_name=input_file_name)
         completeness_results = EvaluateCompleteness(
             clusters_true, clusters_reco_for_pairing, event_key,
             radius_completeness, min_recopoints_threshold)
         purity_results = EvaluatePurity(
             clusters_true, clusters_reco_for_pairing, event_key,
             radius_purity_xz, radius_purity_yz, radius_purity_xy,
             min_projections=purity_min_projections)
         event_matched_pairs = MatchTrueToReco1to1(completeness_results, purity_results)
         event_pair_metadata_list = add_metadata_true_reco_pair_cluster(
             event_matched_pairs, cluster_category_results,
             file_name=input_file_name, event=evt, apa="Combined", view=view, event_key=event_key)
        # None for both metrics when the pairing did not run, which is what the
        # quality selectors already treat as "not well reconstructed".
        attach_pair_metrics(event_true_var_records, event_pair_metadata_list)

        # ------------------------------------------------------------------
        # SELECTION PERFORMANCE: label every selected reco cluster.
        # Categorised against the SAME reco selection the pairing used, so a
        # cluster is never labelled by an overlap the plots do not show.
        # ------------------------------------------------------------------
        if b_draw_selection_performance:
            event_selection_records = categorize_reco_clusters(
                event_reco_var_records[RECO_SELECTION_BEAM],
                purity_results, completeness_results, event_true_var_records)
            job_selection_records.extend(event_selection_records)

            # Which clusters the tagger cut WOULD remove -- asked, not applied,
            # because this notebook is about that decision rather than its effect.
            beam_clusters = event_reco_clusters[RECO_SELECTION_BEAM]
            n_beam_window_clusters += len(beam_clusters)
            tagged_by_cluster = tag_reco_clusters(result.get('taggers'), beam_clusters)

            # Every beam-window cluster matched to a true neutrino, with the
            # pipeline's own radii and purity bar. Run on the PRE-cut set: the
            # removed clusters are absent from the post-cut one.
            neutrino_matches = match_reco_to_true_neutrino(
                purity_results, completeness_results)
            n_matched_neutrino += len(neutrino_matches)

            invol_removed_entries.extend(save_event_tagger_impact(
                neutrino_matches, tagged_by_cluster, beam_clusters, clusters_true,
                event_vertex_records, output_dir, event_key,
                want_removed=True, want_in_volume=True,
                dir_name=IN_VOLUME_SIGNAL_REMOVED_DIR_NAME,
                title="REMOVED by the cosmic tagger cut -- in-volume neutrino",
                name_prefix="invol_signal_removed",
                # CATEGORISED records: only these carry reco_energy_mev.
                reco_var_records=event_selection_records,
                bee_links=bee_links))

        # ------------------------------------------------------------------
        # AGGREGATE TO JOB LEVEL
        # ------------------------------------------------------------------
        job_true_var_records.extend(event_true_var_records)
        # Flash records for EVERY clustering cluster, before any selection: the
        # beam-window cut is applied to the reco POINTS, not to these records, so
        # this list stays the unselected flash population -- in whichever id
        # namespace RECO_ID_FIELD selects.
        job_img_cluster_flash_records.extend(event_flash_records_for_plot)
        for selection_label in RECO_SELECTION_LABELS:
            job_reco_var_records[selection_label].extend(event_reco_var_records[selection_label])
        for selection_label, _stage in RECO_FLOW_STAGES:
            job_reco_flow_counts[selection_label] += len(event_reco_clusters[selection_label])
        job_cluster_type_records.extend(event_cluster_type_records)
        job_vertex_records.extend(event_vertex_records)

        n_true_neutrino = sum(1 for r in event_true_var_records if r['is_neutrino'])
        n_numu_cc_in    = sum(1 for r in event_true_var_records
                              if r.get('interaction_channel') == 'numu_CC'
                              and r.get('vertex_in_volume') is True)
        print(
            f"  Event {evt}: "
            f"true clusters={len(event_true_var_records)} (neutrino={n_true_neutrino}), "
            f"numu CC in volume={n_numu_cc_in}, "
            f"reco clusters={len(event_reco_var_records[RECO_SELECTION_NOCUTS])} "
            f"(in beam window={len(event_reco_var_records[RECO_SELECTION_BEAM])}), "
            f"1-to-1 pairs={len(event_pair_metadata_list)}, "
            f"neutrino interactions in mc={len(event_vertex_records)}"
        )
        total_events_processed += 1


In [ ]:
# ============================================================================
# INDEX -- one row per figure, plus the clickable link file
# ============================================================================
print(f"\n{'='*70}")
print(f"JOB SUMMARY: {total_files_processed} file(s), {total_events_processed} event(s) processed")
print(f"{'='*70}")

# ALWAYS one BEE set for this whole population -- the nuecc sample has no
# per-chunk BEE sets, so building one set from the population's own events is
# the only way its figures get a clickable link. Uploaded here; fills
# entry['bee_url'] so the link file(s) below carry real per-event urls.
_pop_bee_url = None
if SAMPLE == "nuecc" and invol_removed_entries:
    from build_bee_set_from_links import build_population_bee_set
    _pop_bee_url = build_population_bee_set(
        invol_removed_entries, NUECC_STAGING_ROOT,
        output_dir / IN_VOLUME_SIGNAL_REMOVED_DIR_NAME / "bee_set", IN_VOLUME_SIGNAL_REMOVED_DIR_NAME)
    print(f"  population BEE set: {{_pop_bee_url or 'none / upload failed'}}")

# One BEE link file per interaction channel, one directory above the chunks.
bee_paths = write_bee_links_by_category(invol_removed_entries, output_dir, IN_VOLUME_SIGNAL_REMOVED_DIR_NAME,
                                       bee_set_url=_pop_bee_url)
index_path = write_impact_summary(
    invol_removed_entries, output_dir, IN_VOLUME_SIGNAL_REMOVED_DIR_NAME,
    filename='invol_signal_removed_summary.txt',
    headline="SIGNAL REMOVED BY THE COSMIC TAGGER CUT",
    explanation=[
        "In-volume true neutrinos whose reco cluster survived the beam-window cut",
        "and was then REMOVED by the cosmic tagger cut. This is what the cut cost.",
        "",
        "The cut is PER-FLASH: a cluster is here because a tagger flagged it,",
        "or because it shared that cluster's flash. A neutrino at a DIFFERENT",
        "flash time in the same event is NOT removed.",
    ],
    n_matched_total=n_matched_neutrino,
    n_beam_window_clusters=n_beam_window_clusters,
    n_events=total_events_processed)
print(f"  figures drawn: {len(invol_removed_entries)} of {n_matched_neutrino} "
      f"neutrino-matched cluster(s), from {n_beam_window_clusters} in-beam")
for channel in CHANNELS:
    n = sum(1 for e in invol_removed_entries if e['channel'] == channel)
    print(f"    {channel:<10s} {n:4d}   bee links: {bee_paths[channel].name}")
print(f"  selection: in-volume neutrino match, REMOVED by the cosmic tagger cut")
print(f"  index:     {index_path}")
print(f"  bee links: {index_path.parent / 'bee_links.txt'}")
print(f"  figures:   {index_path.parent}")
